# Entropy Surface Experiment

## Imports

In [1]:
import numpy as np
import znnl as nl

import pandas as pd

from flax import linen as nn
import optax

import matplotlib.pyplot as plt
from neural_tangents import stax

import h5py as hf

import seaborn as sns

Using backend: cpu

Available hardware:

TFRT_CPU_0

## Helper Functions

In [2]:
def build_model(
    input_shape: tuple, 
    output_size: int, 
    n_layers: int, 
    layer_width: int, 
    flatten: bool,
    accuracy: bool,
    parametrization: str,
    batch_size: int
):
    """
    Build a model instance with the given parameters.
    
    Parameters
    ----------
    input_shape : tuple
    output_size : int
    n_layers : int
    layer_width : intind
    flatten : bool
    accuracy : bool
    batch-size : int
    """
    layers = []
    if flatten:
        layers.append(stax.Flatten())
    
    for _ in range(n_layers):
        layers.append(stax.Dense(layer_width, b_std=0.1))
        layers.append(stax.Relu())
        
    layers.append(stax.Dense(output_size, b_std=0.1))
    
    dense_network = stax.serial(*layers)
    
    if accuracy:
        accuracy_fn = znrnd.accuracy_functions.LabelAccuracy()
    else:
        accuracy_fn = None
    
    model = znrnd.models.NTModel(
        loss_fn=znrnd.loss_functions.MeanPowerLoss(order=2),
        optimizer=optax.adam(learning_rate=0.01),
        input_shape=(1, 28, 28, 1),
        nt_module=dense_network,
        accuracy_fn=accuracy_fn,
        batch_size=batch_size,
        training_threshold=0.01
    )
    
    return model

In [3]:
def build_convolutional_model(
    input_shape: tuple, 
    output_size: int, 
    n_layers: int, 
    layer_width: int, 
    flatten: bool,
    accuracy: bool,
    batch_size: int
):
    """
    Build a model instance with the given parameters.
    
    Parameters
    ----------
    input_shape : tuple
    output_size : int
    n_layers : int
    layer_width : intind
    flatten : bool
    accuracy : bool
    batch-size : int
    """
    layers = []
    if flatten:
        pass
    feature_widths = [32, 64]
    for i in range(n_layers):
        layers.append(stax.Conv(feature_widths[i], filter_shape=(4, 4), b_std=0.1))
        layers.append(stax.Relu())
        layers.append(stax.AvgPool(window_shape=(4, 4), strides=(2, 2)))
        
        
    layers.append(stax.Flatten())
    layers.append(stax.Dense(layer_width, b_std=0.1))
    layers.append(stax.Relu())
    layers.append(stax.Dense(output_size, b_std=0.1))
    
    dense_network = stax.serial(*layers)
    
    if accuracy:
        accuracy_fn = znrnd.accuracy_functions.LabelAccuracy()
    else:
        accuracy_fn = None
    
    model = znrnd.models.NTModel(
        loss_fn=znrnd.loss_functions.MeanPowerLoss(order=2),
        optimizer=optax.adam(learning_rate=0.01),
        input_shape=(1, 28, 28, 1),
        nt_module=dense_network,
        accuracy_fn=accuracy_fn,
        batch_size=batch_size,
        training_threshold=0.01
    )
    
    return model

## Run the experiment

In [8]:
generator = nl.data.MNISTGenerator(ds_size=500)

In [9]:
ensembles = 10

for i in range(ensembles):
    network = stax.serial(
        stax.Flatten(),
        stax.Dense(128, b_std=0.01, parameterization="standard"),
        stax.Relu(),
        stax.Dense(128, b_std=0.01, parameterization="standard"),
        stax.Relu(),
        stax.Dense(10, b_std=0.01, parameterization="standard"),
    )

    optimizer = optax.adam(1e-3)

    model = nl.models.NTModel(
            nt_module=network,
            optimizer=optimizer,
            input_shape=(1, 28, 28, 1),
    )
    cv_recorder = nl.training_recording.JaxRecorder(
        name=f"cv_train_recorder_{i}",
        entropy=True,
        trace=True,
        update_rate=500
    )
    train_recorder = nl.training_recording.JaxRecorder(
        name=f"adam_train_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1
    )
    test_recorder = nl.training_recording.JaxRecorder(
        name=f"adam_test_recorder_{i}",
        loss=True,
        accuracy=True,
        update_rate=1
    )
    
    train_recorder.instantiate_recorder(
        data_set=generator.train_ds
    )
    cv_recorder.instantiate_recorder(
        data_set=generator.train_ds
    )
    test_recorder.instantiate_recorder(
        data_set=generator.test_ds
    )
    
    training_strategy = nl.training_strategies.SimpleTraining(
        model=model, 
        loss_fn=nl.loss_functions.MeanPowerLoss(order=2),
        accuracy_fn=nl.accuracy_functions.LabelAccuracy(),
        recorders=[train_recorder, test_recorder, cv_recorder],
    )
    _ = training_strategy.train_model(
            train_ds=generator.train_ds,
            test_ds=generator.test_ds, 
            epochs=100, 
            batch_size=32,
#             recorders=[train_recorder, test_recorder],
        )
    
    train_recorder.dump_records()
    test_recorder.dump_records()
    cv_recorder.dump_records()

  0%|                                                                    | 0/100 [00:00<?, ?batch/s]

adam_train_recorder_0 is recording epoch 0
adam_test_recorder_0 is recording epoch 0
cv_train_recorder_0 is recording epoch 0


2023-07-21 00:02:17.122756: E external/xla/xla/service/slow_operation_alarm.cc:65] 
********************************
[Compiling module jit_ntk_fn] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
2023-07-21 00:02:57.611367: E external/xla/xla/service/slow_operation_alarm.cc:133] The operation took 2m40.491543s

********************************
[Compiling module jit_ntk_fn] Very slow compile? If you want to file a bug, run with envvar XLA_FLAGS=--xla_dump_to=/tmp/foo and attach the results.
********************************
Epoch: 1:   1%|▎                               | 1/100 [11:03<18:14:58, 663.62s/batch, accuracy=0.7]

adam_train_recorder_0 is recording epoch 1
adam_test_recorder_0 is recording epoch 1


Epoch: 2:   2%|▌                              | 2/100 [11:03<7:26:33, 273.41s/batch, accuracy=0.814]

...

Epoch: 99:  99%|████████████████████████████████▋| 99/100 [11:28<00:00,  4.05batch/s, accuracy=0.87]

adam_train_recorder_0 is recording epoch 99
adam_test_recorder_0 is recording epoch 99


  0%|                                                                    | 0/100 [00:00<?, ?batch/s]

adam_train_recorder_1 is recording epoch 0
adam_test_recorder_1 is recording epoch 0
cv_train_recorder_1 is recording epoch 0


  0%|                                                                    | 0/100 [02:03<?, ?batch/s]

KeyboardInterrupt

